# 3주차 실습: 한국어 텍스트 전처리

**제출용** — 필수 TODO 1~10과 체크포인트 답변을 작성했습니다. 선택 심화 1~3과 실제 리뷰 적용 결과도 포함했습니다.

선택 심화 3에는 공개 NSMC 자료에서 실제 네이버 영화 리뷰 5건을 선별해 사용했습니다. 원문 사이트에서 새로 크롤링한 자료는 아닙니다.


## 0. 준비 및 실습 데이터

이 데이터는 교육용 예시입니다. 실제 리뷰·게시글을 분석할 때에는 데이터 출처·이용 조건·개인정보 포함 여부를 확인해야 합니다.

In [1]:
# 공용 NLP 커널에 설치된 라이브러리 버전을 확인합니다.
# 다른 환경에서 패키지가 없다면 다음 줄을 실행한 뒤 커널을 재시작하세요.
# %pip install kiwipiepy konlpy pandas
from importlib.metadata import version
for package in ['kiwipiepy', 'konlpy', 'pandas']:
    print(f'{package}: {version(package)}')

kiwipiepy: 0.23.2
konlpy: 0.6.0
pandas: 3.0.6


In [2]:

import re
import unicodedata
from collections import Counter

import pandas as pd
from kiwipiepy import Kiwi

kiwi = Kiwi()

# Okt는 Java 설정에 따라 초기화가 실패할 수 있습니다. 실패해도 Kiwi 실습은 계속할 수 있습니다.
try:
    from konlpy.tag import Okt
    okt = Okt()
    okt_ready = True
except Exception as error:
    okt = None
    okt_ready = False
    print('Okt를 사용할 수 없습니다. Kiwi 중심으로 진행합니다.')
    print('사유:', type(error).__name__)

reviews = [
    (1, "정말 재미있어요!!! 배우 연기가 최고네요 😊", 1),
    (2, "스토리는 평범하지만 음악이 좋아요.", 1),
    (3, "기대했는데 너무너무 지루했어요...", 0),
    (4, "영상미는 훌륭합니다. 그런데 전개가 느려요.", 1),
    (5, "별로예요 ㅠㅠ 다시 보고 싶지 않아요.", 0),
    (6, "문의는 movie@example.com 또는 010-1234-5678로 주세요.", 1),
    (7, "새 소식: https://example.com/movie 를 확인하세요!", 1),
    (8, "가족과 함께 보기 좋은 따뜻한 작품입니다.", 1),
]
df = pd.DataFrame(reviews, columns=['review_id', 'text', 'label'])
df['label_name'] = df['label'].map({1: '긍정', 0: '부정'})
display(df)

,review_id,text,label,label_name
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,1,긍정
1,2,스토리는 평범하지만 음악이 좋아요.,1,긍정
2,3,기대했는데 너무너무 지루했어요...,0,부정
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,1,긍정
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,0,부정
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,1,긍정
6,7,새 소식: https://example.com/movie 를 확인하세요!,1,긍정
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,1,긍정


## 1. 전처리 파이프라인 이해

전처리는 하나의 정답 함수가 아닙니다. **과업·데이터·평가 기준**에 따라 필요한 규칙이 달라집니다.

`원문 → 유니코드 정규화 → 민감정보/노이즈 탐지 → 처리 정책(보존·마스킹·삭제) → 토큰화 → 품사·불용어 필터 → 텍스트 표현`

- URL·연락처는 단순 노이즈가 아니라 개인정보 또는 참조 정보일 수 있습니다.
- 이모지·반복 문자·느낌표는 감성 신호일 수 있으므로 무조건 삭제하지 않습니다.
- 이후 주차의 빈도·TF-IDF는 여기서 만든 토큰을 입력으로 사용합니다.

### 체크포인트 1

위 파이프라인에서 **반드시 보존해야 할 수 있다고 생각하는 정보**를 하나 고르고, 어떤 분석 과업에서 유용한지 1~2문장으로 설명하세요.

**답변**

부정 표현인 “않다”는 감성 분석에서 보존해야 한다. “좋아요”와 “좋지 않아요”의 극성을 구분하는 핵심 정보이므로, 불용어 처리나 품사 필터에서 제거하면 의미가 뒤바뀔 수 있다.

## 2. 유니코드 정규화: 겉보기와 내부 표현

겉보기에는 같은 한글도 내부 유니코드 표현이 달라 비교·중복 제거에 문제가 생길 수 있습니다. 분석 전에는 보통 NFC 형태로 통일합니다.

In [3]:
composed = '가'
decomposed = unicodedata.normalize('NFD', composed)
print('겉보기:', composed, decomposed)
print('정규화 전 동일 여부:', composed == decomposed)

# TODO 1 답안
nfc_composed = unicodedata.normalize('NFC', composed)
nfc_decomposed = unicodedata.normalize('NFC', decomposed)
print('NFC 정규화 후 동일 여부:', nfc_composed == nfc_decomposed)

겉보기: 가 가
정규화 전 동일 여부: False
NFC 정규화 후 동일 여부: True


In [4]:
# TODO 2 답안
df['text_nfc'] = df['text'].apply(lambda text: unicodedata.normalize('NFC', text))
display(df[['review_id', 'text', 'text_nfc']])

,review_id,text,text_nfc
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊
1,2,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.
2,3,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 movie@example.com 또는 010-1234-5678로 주세요.
6,7,새 소식: https://example.com/movie 를 확인하세요!,새 소식: https://example.com/movie 를 확인하세요!
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.


### 체크포인트 2

NFC 정규화를 하지 않으면 **중복 제거**, **검색**, **모델 입력** 중 하나에서 어떤 문제가 발생할 수 있는지 예를 들어 설명하세요.

**답변**

같은 “가”도 NFC에서는 한 글자(U+AC00), NFD에서는 두 자모(U+1100, U+1161)로 저장된다. 정규화하지 않으면 문자열 비교가 달라져 같은 리뷰가 중복 제거에서 남거나, “가”를 검색했을 때 자모로 저장된 텍스트가 누락될 수 있다.

## 3. 노이즈·개인정보 패턴 탐지와 처리 정책

정규표현식은 일정한 형식의 문자열을 찾는 도구입니다. 아래 탐지는 교육용 예시이며, 실제 개인정보 탐지에는 더 엄격한 정책·검토가 필요합니다. 실습에서는 연락처를 즉시 삭제하지 않고 **마스킹**해 정보의 유형을 남깁니다.

In [5]:
def inspect_patterns(text):
    flags = []
    if re.search(r'https?://\S+|www\.\S+', text):
        flags.append('URL')
    if re.search(r'[\w.+-]+@[\w-]+\.[\w.-]+', text):
        flags.append('이메일 가능성')
    # TODO 3 답안
    if re.search(r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}', text):
        flags.append('전화번호 가능성')
    # TODO 4 답안: 반복 구간의 존재를 탐지합니다.
    if re.search(r'(.)\1{2,}', text):
        flags.append('동일 문자 3회 이상 반복')
    return ', '.join(flags) if flags else '특이 패턴 없음'

df['pattern_flags'] = df['text_nfc'].apply(inspect_patterns)
display(df[['review_id', 'text_nfc', 'pattern_flags']])

,review_id,text_nfc,pattern_flags
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,동일 문자 3회 이상 반복
1,2,스토리는 평범하지만 음악이 좋아요.,특이 패턴 없음
2,3,기대했는데 너무너무 지루했어요...,동일 문자 3회 이상 반복
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,특이 패턴 없음
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,특이 패턴 없음
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,"이메일 가능성, 전화번호 가능성"
6,7,새 소식: https://example.com/movie 를 확인하세요!,URL
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,특이 패턴 없음


In [6]:
def basic_clean(text, mask_sensitive=True):
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' [URL] ' if mask_sensitive else ' ', text)
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' [EMAIL] ' if mask_sensitive else ' ', text)
    text = re.sub(r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}', ' [PHONE] ' if mask_sensitive else ' ', text)
    # TODO 5 답안
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text_nfc'].apply(basic_clean)
display(df[['text_nfc', 'clean_text']])
print('마스킹:', basic_clean(df.loc[5, 'text'], mask_sensitive=True))
print('삭제  :', basic_clean(df.loc[5, 'text'], mask_sensitive=False))

,text_nfc,clean_text
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊
1,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.
2,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.
5,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 [EMAIL] 또는 [PHONE] 로 주세요.
6,새 소식: https://example.com/movie 를 확인하세요!,새 소식: [URL] 를 확인하세요!
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.


마스킹: 문의는 [EMAIL] 또는 [PHONE] 로 주세요.
삭제  : 문의는 또는 로 주세요.


### 체크포인트 3

반복 문자 또는 이모지를 일괄 삭제할 때 생길 수 있는 **정보 손실** 사례를 한 가지 작성하세요. 전화번호·이메일을 탐지한 뒤 실제 서비스에서 추가로 고려해야 할 조치도 한 가지 적으세요.

**답변**

“정말 재미있어요!!! 😊”에서 느낌표 반복과 이모지를 삭제하면 긍정 감정의 강도와 표정 단서가 사라진다. 연락처는 [PHONE], [EMAIL]로 마스킹하면 종류와 등장 여부를 남기지만, 삭제하면 그 단서까지 사라진다. 실제 서비스에서는 정규표현식이 놓치는 형식과 오탐을 점검하고, 원문 연락처가 로그나 공개 결과에 남지 않도록 접근 권한과 보관 정책을 적용해야 한다.

## 4. 토큰화 비교: 공백·정규표현식·형태소 분석

공백 기준 토큰화는 간단하지만 조사·어미가 결합된 한국어의 문법 정보를 충분히 분리하지 못합니다. Okt가 준비되지 않은 경우에도 Kiwi 결과와 공백 토큰을 비교해 관찰하세요.

In [7]:
if 'clean_text' not in df.columns:
    print('TODO 5를 완료한 뒤 다시 실행하세요.')
else:
    sample = df.loc[0, 'clean_text']
    print('원문:', sample)
    print('공백 토큰:', sample.split())
    print('정규표현식 토큰:', re.findall(r'[가-힣A-Za-z0-9]+', sample))
    print('\nKiwi 형태소:', [(token.form, token.tag) for token in kiwi.tokenize(sample)])
    if okt_ready:
        print('Okt 형태소:', okt.pos(sample))
    else:
        print('Okt 형태소: 현재 환경에서는 비교 생략')

원문: 정말 재미있어요!!! 배우 연기가 최고네요 😊
공백 토큰: ['정말', '재미있어요!!!', '배우', '연기가', '최고네요', '😊']
정규표현식 토큰: ['정말', '재미있어요', '배우', '연기가', '최고네요']

Kiwi 형태소: [('정말', 'MAG'), ('재미있', 'VA'), ('어요', 'EF'), ('!!!', 'SF'), ('배우', 'NNG'), ('연기', 'NNG'), ('가', 'JKS'), ('최고', 'NNG'), ('이', 'VCP'), ('네요', 'EF'), ('😊', 'W_EMOJI')]
Okt 형태소: [('정말', 'Noun'), ('재미있어요', 'Adjective'), ('!!!', 'Punctuation'), ('배우', 'Noun'), ('연기', 'Noun'), ('가', 'Josa'), ('최고', 'Noun'), ('네', 'Suffix'), ('요', 'Josa'), ('😊', 'Foreign')]


In [8]:
# TODO 6 답안: 기존에 선택한 맥북 주제 문장을 사용합니다.
my_sentence = '맥북 프로는 전문 개발자 혹은 영상 편집 및 음악 프로듀서들 사이에서 꾸준한 호평을 받고 있는 애플의 고사양 스테디셀러 제품군이다.'
print('문장:', my_sentence)
my_kiwi_tokens = [(t.form, t.tag) for t in kiwi.tokenize(my_sentence)]
print('Kiwi:', my_kiwi_tokens)
if okt_ready:
    my_okt_tokens = okt.pos(my_sentence)
    print('Okt :', my_okt_tokens)
    common_nouns = [(form, tag, okt_tag)
                    for form, tag in my_kiwi_tokens if tag.startswith('N')
                    for okt_form, okt_tag in my_okt_tokens
                    if form == okt_form and okt_tag == 'Noun']
    print('동일 표면형의 품사 태그 비교:', common_nouns[:5])
else:
    print('Okt : 현재 환경에서는 비교 생략')

문장: 맥북 프로는 전문 개발자 혹은 영상 편집 및 음악 프로듀서들 사이에서 꾸준한 호평을 받고 있는 애플의 고사양 스테디셀러 제품군이다.
Kiwi: [('맥북', 'NNP'), ('프로', 'NNG'), ('는', 'JX'), ('전문', 'NNG'), ('개발자', 'NNG'), ('혹은', 'MAG'), ('영상', 'NNG'), ('편집', 'NNG'), ('및', 'MAG'), ('음악', 'NNG'), ('프로듀서', 'NNG'), ('들', 'XSN'), ('사이', 'NNG'), ('에서', 'JKB'), ('꾸준하', 'VA'), ('ᆫ', 'ETM'), ('호평', 'NNG'), ('을', 'JKO'), ('받', 'VV-R'), ('고', 'EC'), ('있', 'VX'), ('는', 'ETM'), ('애플', 'NNP'), ('의', 'JKG'), ('고', 'XPN'), ('사양', 'NNG'), ('스테디셀러', 'NNG'), ('제품군', 'NNG'), ('이', 'VCP'), ('다', 'EF'), ('.', 'SF')]
Okt : [('맥북', 'Noun'), ('프로', 'Noun'), ('는', 'Josa'), ('전문', 'Noun'), ('개발자', 'Noun'), ('혹은', 'Adverb'), ('영상', 'Noun'), ('편집', 'Noun'), ('및', 'Noun'), ('음악', 'Noun'), ('프로듀서', 'Noun'), ('들', 'Suffix'), ('사이', 'Noun'), ('에서', 'Josa'), ('꾸준한', 'Adjective'), ('호평', 'Noun'), ('을', 'Josa'), ('받고', 'Verb'), ('있는', 'Adjective'), ('애플', 'Noun'), ('의', 'Josa'), ('고', 'Modifier'), ('사양', 'Noun'), ('스', 'Noun'), ('테디', 'Noun'), ('셀러', 'Noun'), ('제품군', 'Noun'), ('이다',

### 체크포인트 4

본인 문장에서 Kiwi와 Okt가 다르게 나눈 토큰 또는 다르게 태깅한 사례를 하나 기록하세요. Okt를 사용할 수 없었다면 Kiwi 결과에서 조사·어미가 어떻게 분리되었는지 기록하세요. 결과의 우열이 아니라, **어떤 과업에 더 편리할지**를 함께 설명하세요.

**답변**

‘스테디셀러’의 경우 Kiwi는 하나의 명사 토큰으로 분석했지만, Okt는 ‘스’, ‘테디’, ‘셀러’로 나누어 분석했다. 제품 리뷰에서 주요 키워드를 추출하는 목적이라면 ‘스테디셀러’를 하나의 의미 단위로 유지하는 Kiwi의 결과가 더 편리하다고 생각한다.


## 5. 품사 선택과 불용어 처리

키워드 추출에는 명사 중심 토큰이 유용할 수 있습니다. 반면 감성 분석에서는 부정 표현·부사·어미가 중요할 수 있습니다. 아래에서는 먼저 명사를 추출한 뒤, **도메인 불용어**를 적용합니다. 조사는 이미 품사 필터에서 제거되므로, 명사 목록에 조사 불용어를 넣어도 효과가 거의 없다는 점을 확인하세요.

In [9]:
def kiwi_nouns(text):
    # TODO 7 답안
    return [token.form for token in kiwi.tokenize(text) if token.tag.startswith('N')]

df['nouns'] = df['clean_text'].apply(kiwi_nouns)
display(df[['review_id', 'clean_text', 'nouns']])

,review_id,clean_text,nouns
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]"
1,2,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]"
2,3,기대했는데 너무너무 지루했어요...,[기대]
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]"
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[]
5,6,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의]
6,7,새 소식: [URL] 를 확인하세요!,"[소식, 확인]"
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,"[가족, 작품]"


In [10]:
stopwords = {'영화', '작품'}
# TODO 8 답안
df['filtered_nouns'] = df['nouns'].apply(
    lambda tokens: [token for token in tokens if token not in stopwords]
)
display(df[['clean_text', 'nouns', 'filtered_nouns']])

,clean_text,nouns,filtered_nouns
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]","[배우, 연기, 최고]"
1,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]","[스토리, 음악]"
2,기대했는데 너무너무 지루했어요...,[기대],[기대]
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]","[영상미, 전개]"
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[],[]
5,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의],[문의]
6,새 소식: [URL] 를 확인하세요!,"[소식, 확인]","[소식, 확인]"
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,"[가족, 작품]",[가족]


In [11]:
# TODO 9 답안
all_nouns = [token for tokens in df['filtered_nouns'] for token in tokens]
noun_freq = Counter(all_nouns)
display(pd.DataFrame(noun_freq.most_common(10), columns=['token', 'count']))

,token,count
0,배우,1
1,연기,1
2,최고,1
3,스토리,1
4,음악,1
5,기대,1
6,영상미,1
7,전개,1
8,문의,1
9,소식,1


### 체크포인트 5

`좋지 않아요`, `너무 지루했어요`와 같은 감성 표현을 생각해 보세요. 명사만 남기면 어떤 정보가 사라질 수 있나요? 감성 분석과 키워드 추출 중 어느 과업에 명사 중심 처리가 더 적합한지 이유와 함께 답하세요.

**답변**

명사만 남기면 “좋다”, “지루하다” 같은 평가 표현과 “않다” 같은 부정 표현, “너무” 같은 강도 부사가 사라질 수 있다. 따라서 명사 중심 처리는 대상·주제를 찾는 키워드 추출에 더 적합하며, 감성 분석에는 용언·부정·부사와 문맥을 함께 보존하는 편이 적절하다. 도메인 불용어인 “작품”도 분석 목적이 바뀌면 유용할 수 있으므로 항상 제거할 단어는 아니다.

## 6. 전처리 전·후 비교

전처리의 효과는 단순히 “짧아졌다”가 아니라, 과업에 필요한 정보가 적절히 남았는지로 판단해야 합니다.

In [12]:
# TODO 10의 판단 근거가 되는 전처리 전·후 비교
comparison = df[['review_id', 'text', 'clean_text', 'filtered_nouns']].copy()
comparison['original_length'] = comparison['text'].str.len()
comparison['clean_length'] = comparison['clean_text'].str.len()
display(comparison)
display(comparison.loc[comparison['review_id'] == 5])

,review_id,text,clean_text,filtered_nouns,original_length,clean_length
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]",25,25
1,2,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]",19,19
2,3,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...,[기대],19,19
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]",24,24
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[],21,21
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의],44,29
6,7,새 소식: https://example.com/movie 를 확인하세요!,새 소식: [URL] 를 확인하세요!,"[소식, 확인]",40,20
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.,[가족],23,23


,review_id,text,clean_text,filtered_nouns,original_length,clean_length
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[],21,21


### TODO 10 답변 — 전처리 전·후 판단

1. 5번 리뷰인 “별로예요 ㅠㅠ 다시 보고 싶지 않아요.”는 `basic_clean` 단계에서는 내용이 거의 그대로 보존되지만, 명사만 추출한 `filtered_nouns`에서는 빈 리스트가 된다. 즉, 부정적인 감정을 나타내는 표현들이 명사 필터 과정에서 사라진다.

2. 이러한 변화는 명사 중심 키워드 추출에는 사용할 수 있지만 감성 분석에는 적합하지 않다. 이 리뷰의 핵심 정보는 “별로예요”, “ㅠㅠ”, “싶지 않아요”와 같은 부정적인 감정 표현에 있기 때문이다.

3. 따라서 감성 분석을 목적으로 한다면 “별로예요”, “ㅠㅠ”, “싶지 않아요”와 같은 감성 표현을 삭제하지 않고 보존할 필요가 있다.


## 제출 체크리스트

- [x] TODO 1~10을 작성하고 실행했다.
- [x] NFC 정규화의 필요성을 설명했다.
- [x] URL·이메일·전화번호·반복 문자 중 2개 이상을 탐지했다.
- [x] 민감정보를 마스킹할 때와 삭제할 때의 차이를 설명했다.
- [x] 공백·정규표현식·Kiwi·(가능하면) Okt 토큰화 결과를 비교했다.
- [x] 명사 중심 처리와 도메인 불용어 처리의 한계를 설명했다.
- [x] 전처리 전·후 비교 결과와 판단 근거를 기록했다.
- [x] 외부 AI/자료 사용 시 사용 범위와 검증 과정을 기록했다.

### 심화 과제(선택)
1. `basic_clean`에서 이모지·반복 문자 수를 별도 특징으로 보존하세요.
2. 감성 표현을 보존하려면 명사 외에 어떤 품사를 남겨야 하는지 규칙을 제안·구현하세요.
3. 직접 수집한 텍스트 5건 이상에 같은 규칙을 적용하고, 출처·라이선스·개인정보 처리 방침을 기록하세요.

## 다음 주차 예고

다음 주차에는 전처리된 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터로 표현합니다. 단어 빈도, Bag-of-Words, TF-IDF를 이용해 문서를 비교하고 분류 모델의 입력 특징을 구성합니다.

## 선택 심화 1~2 — 감성 특징과 감성용 토큰

정제 전 NFC 원문에서 이모지 수, 같은 문자가 3회 이상 반복되는 구간 수와 해당 구간의 문자 수를 따로 보존한다. 이모지는 아래 범위에 해당하는 코드 포인트를 세므로, 결합 이모지의 화면상 개수와 다를 수 있는 교육용 근사치다. 감성용 토큰에는 명사·용언·부사·보조용언·어미·접두/접미사를 남겨 부정·강도·시제 정보를 보존한다. 토큰만으로 부정의 적용 범위를 완전히 해석할 수는 없으므로 원문도 함께 유지한다.

In [13]:
EMOJI_PATTERN = re.compile('[\U0001F300-\U0001FAFF\u2600-\u27BF]')

def clean_with_features(text):
    normalized = unicodedata.normalize('NFC', text)
    repeats = [match.group(0) for match in re.finditer(r'(.)\1{2,}', normalized)]
    return {
        'clean_text': basic_clean(normalized),
        'emoji_count': len(EMOJI_PATTERN.findall(normalized)),
        'repeat_group_count': len(repeats),
        'repeated_char_count': sum(map(len, repeats)),
    }

def sentiment_tokens(text):
    # 마스킹 태그를 키워드/감성 단어로 해석하지 않도록 제외합니다.
    text = re.sub(r'\[(?:URL|EMAIL|PHONE)\]', ' ', text)
    return [(t.form, t.tag) for t in kiwi.tokenize(text)
            if t.tag.startswith(('N', 'V', 'M', 'E', 'X'))]

features = pd.DataFrame(df['text_nfc'].apply(clean_with_features).tolist())
display(pd.concat([df[['review_id', 'text']], features], axis=1))
for text in ['좋지 않아요', '너무 지루했어요']:
    print(text)
    print('명사만:', kiwi_nouns(text))
    print('감성용:', sentiment_tokens(text))

,review_id,text,clean_text,emoji_count,repeat_group_count,repeated_char_count
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊,1,1,3
1,2,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.,0,0,0
2,3,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...,0,1,3
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.,0,0,0
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,0,0,0
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,0,0,0
6,7,새 소식: https://example.com/movie 를 확인하세요!,새 소식: [URL] 를 확인하세요!,0,0,0
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.,0,0,0


좋지 않아요
명사만: []
감성용: [('좋', 'VA'), ('지', 'EC'), ('않', 'VX'), ('어요', 'EF')]
너무 지루했어요
명사만: []
감성용: [('너무', 'MAG'), ('지루', 'XR'), ('하', 'XSA'), ('었', 'EP'), ('어요', 'EF')]


## 선택 심화 3 — 실제 영화 리뷰 5건 수집 및 적용

공개된 **Naver Sentiment Movie Corpus(NSMC)**의 `ratings_test.txt`에서 실제 리뷰 5건을 확보했다. 긍정 3건과 부정 2건을 골라 반복 부호·띄어쓰기·부정 표현의 처리 결과를 비교한다. 생성한 문장으로 대체하지 않았으며, 오탈자와 문장부호를 그대로 보존했다. 소수의 목적 표본이므로 전체 리뷰 분포를 대표하지 않는다.

- 원출처: 네이버 영화 사용자 리뷰. NSMC 작성자가 네이버 영화에서 수집하여 공개한 자료이며, 이번 실습에서는 해당 공개 코퍼스에서 5건을 선별해 사용했다.
- 제공처: [e9t/nsmc](https://github.com/e9t/nsmc).
- 고정된 원본 파일: [ratings_test.txt](https://raw.githubusercontent.com/e9t/nsmc/cc0670e872d4ac27bfe36c87456783004b39ef6c/ratings_test.txt). 아래 표의 리뷰 ID 및 원본 행 번호로 확인할 수 있다(헤더를 1행으로 계산).
- 이번 확보 시각: 2026-09-21T12:31:26+09:00. 리뷰 게시일 및 최초 크롤링 날짜는 이 파일에 없어 확인하지 않았다.
- 버전: `cc0670e872d4ac27bfe36c87456783004b39ef6c`. 파일 SHA-256: `8ac9f64052f11dbf6ae0acb5e038f03d90a76f0eda7820cfb3a92d02edfcebda`.
- 라이선스: 저장소 [README](https://github.com/e9t/nsmc/blob/cc0670e872d4ac27bfe36c87456783004b39ef6c/README.md)의 License 항목은 [CC0 1.0](https://creativecommons.org/publicdomain/zero/1.0/)을 표시한다. 해당 배포 표기를 기록하며, 원 리뷰의 모든 권리나 개인정보 문제가 자동으로 해소된다고 해석하지 않는다.
- 개인정보: 선택한 다섯 문장은 육안으로 확인했으며 연락처·이메일·URL·개인 실명은 보이지 않는다. 작성자 계정은 수집하지 않았고, 리뷰 ID는 출처 확인에만 사용한다. 패턴 탐지와 마스킹은 동일하게 적용한다.
- 재현: 확보한 다섯 문장과 ID를 아래 코드에 포함해 네트워크 없이 다시 실행할 수 있다.


In [14]:
collected_reviews = [{'review_id': '7898805', 'text': '음악이 주가 된, 최고의 음악영화', 'label': 1, 'source_line': 7}, {'review_id': '2968565', 'text': '최고!!!!!!!!!!!!!!!!', 'label': 1, 'source_line': 18}, {'review_id': '9727606', 'text': '꽤 재밌게 본 영화였다!', 'label': 1, 'source_line': 24}, {'review_id': '6484287', 'text': '....재미가없어요 시간이 아깝고', 'label': 0, 'source_line': 38}, {'review_id': '7530635', 'text': '완전 잼없음..', 'label': 0, 'source_line': 91}]
extra_df = pd.DataFrame(collected_reviews)
extra_df['label_name'] = extra_df['label'].map({1: '긍정', 0: '부정'})
extra_df['text_nfc'] = extra_df['text'].apply(lambda x: unicodedata.normalize('NFC', x))
extra_df['pattern_flags'] = extra_df['text_nfc'].apply(inspect_patterns)
extra_features = pd.DataFrame(extra_df['text_nfc'].apply(clean_with_features).tolist())
extra_df = pd.concat([extra_df, extra_features], axis=1)
extra_df['nouns'] = extra_df['clean_text'].apply(kiwi_nouns)
extra_df['filtered_nouns'] = extra_df['nouns'].apply(
    lambda tokens: [token for token in tokens if token not in stopwords]
)
extra_df['sentiment_tokens'] = extra_df['clean_text'].apply(sentiment_tokens)
display(extra_df[['review_id', 'source_line', 'text', 'label_name']])
display(extra_df[['review_id', 'clean_text', 'pattern_flags', 'emoji_count',
                 'repeat_group_count', 'repeated_char_count']])
display(extra_df[['review_id', 'nouns', 'filtered_nouns', 'sentiment_tokens']])
extra_frequency = Counter(token for tokens in extra_df['filtered_nouns'] for token in tokens)
display(pd.DataFrame(extra_frequency.most_common(10), columns=['token', 'count']))


,review_id,source_line,text,label_name
0,7898805,7,"음악이 주가 된, 최고의 음악영화",긍정
1,2968565,18,최고!!!!!!!!!!!!!!!!,긍정
2,9727606,24,꽤 재밌게 본 영화였다!,긍정
3,6484287,38,....재미가없어요 시간이 아깝고,부정
4,7530635,91,완전 잼없음..,부정


,review_id,clean_text,pattern_flags,emoji_count,repeat_group_count,repeated_char_count
0,7898805,"음악이 주가 된, 최고의 음악영화",특이 패턴 없음,0,0,0
1,2968565,최고!!!!!!!!!!!!!!!!,동일 문자 3회 이상 반복,0,1,16
2,9727606,꽤 재밌게 본 영화였다!,특이 패턴 없음,0,0,0
3,6484287,....재미가없어요 시간이 아깝고,동일 문자 3회 이상 반복,0,1,4
4,7530635,완전 잼없음..,특이 패턴 없음,0,0,0


,review_id,nouns,filtered_nouns,sentiment_tokens
0,7898805,"[음악, 주, 최고, 음악, 영화]","[음악, 주, 최고, 음악]","[(음악, NNG), (주, NNG), (되, VV), (ᆫ, ETM), (최고, ..."
1,2968565,[최고],[최고],"[(최고, NNG)]"
2,9727606,[영화],[],"[(꽤, MAG), (재밌, VA), (게, EC), (보, VV), (ᆫ, ETM..."
3,6484287,"[재미, 시간]","[재미, 시간]","[(재미, NNG), (없, VA), (어요, EF), (시간, NNG), (아깝,..."
4,7530635,[잼],[잼],"[(완전, MAG), (잼, NNG), (없, VA), (음, ETN)]"


,token,count
0,음악,2
1,최고,2
2,주,1
3,재미,1
4,시간,1
5,잼,1


### 실제 리뷰 적용 결과 해석

- 리뷰 `2968565`의 반복 느낌표는 `basic_clean`에서 유지되고, 반복 구간·문자 수로 별도 기록된다. 명사 목록만으로는 긍정의 강조 강도를 나타내기 어렵다.
- 리뷰 `6484287`에는 반복 마침표와 부정 평가가 있어 반복 패턴 탐지 및 감성용 품사 선택을 함께 비교할 수 있다. 띄어쓰기 오류를 자동 수정하지 않았으므로 분석기의 분절 결과도 관찰 대상이다.
- 리뷰 `7530635`의 구어적 축약 표현은 사전에 따라 분석이 달라질 수 있다. 명사가 적거나 기대한 평가 단어가 추출되지 않았다고 해서 원문에 감성이 없다고 볼 수는 없다.
- 명사 빈도표는 영화의 주제·대상을 살피는 보조 수단이다. 감성 판단에는 원문, 감성용 토큰, 반복 부호 특징을 함께 고려해야 한다. 이 다섯 문장에서 연락처가 발견되지 않는다고 해서 탐지 규칙의 정확성이 검증된 것은 아니다.


## AI 활용 및 실행 기록

**심화 과제에서 AI를 적극 활용했다.** 이모지·반복 문자 특징의 설계, 감성 정보를 보존하는 품사 선택 규칙의 구현, 공개 자료에서 실제 리뷰 5건을 확보하고 출처·라이선스를 정리하는 작업, 전처리 결과 비교에 ChatGPT를 활용했다. 명사 중심 분석이 놓치는 부정 표현과 강조 부호를 함께 관찰할 수 있도록 분석 범위를 확장했다.

**기본 과제(TODO 1~10)와 체크포인트**에도 AI를 활용해 사용자가 작성한 코드와 서술형 답안 초안을 검토하고 오탈자 수정 및 설명 구체화 작업을 수행했다.

**실행 및 한계:** 수정본을 공용 Python 3.12 NLP 커널에서 위에서 아래로 한 번 실행하고 출력을 포함했다. 별도의 반복 검사나 정확도 평가는 수행하지 않았다. 실제 리뷰 5건은 목적 표본이며, 정규표현식과 이모지 계수는 교육용 근사치다.
